# COM6003 CW1

EPC data for Mid Buckinghamshire. Downloaded domestic certificates Jan 2012 - May 2026, ratings A-G.

HamzA Chennou - 22331945

In [5]:
# imports and paths
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

LA_CODE = "E06000060"
LA_NAME = "Mid Buckinghamshire"
TARGET = "CURRENT_ENERGY_EFFICIENCY"
RANDOM_STATE = 42
DATE_START = pd.Timestamp("2012-01-01")
DATE_END = pd.Timestamp("2026-05-31")
RATINGS = ["A", "B", "C", "D", "E", "F", "G"]

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
FIG_DIR = Path("outputs/figures")
for p in [RAW_DIR, PROCESSED_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

RAW_FILE = RAW_DIR / f"{LA_CODE}_certificates.csv"
CLEAN_FILE = PROCESSED_DIR / f"{LA_CODE}_clean.csv"
MODEL_FILE = PROCESSED_DIR / f"{LA_CODE}_model_ready.csv"

In [6]:
# load raw epc data
df_raw = pd.read_csv(RAW_FILE, low_memory=False)

lodgement_col = next(c for c in df_raw.columns if c.lower() == "lodgement_date")
constituency_col = next(c for c in df_raw.columns if c.lower() == "constituency_label")
rating_col_raw = next(c for c in df_raw.columns if c.lower() == "current_energy_rating")

df_raw[lodgement_col] = pd.to_datetime(df_raw[lodgement_col], errors="coerce")
df_raw = df_raw[df_raw[lodgement_col].between(DATE_START, DATE_END)]
df_raw = df_raw[df_raw[constituency_col] == LA_NAME]
df_raw = df_raw[df_raw[rating_col_raw].isin(RATINGS)]

print(len(df_raw), "rows,", df_raw.shape[1], "columns")
print("area:", LA_NAME)
print("date range:", df_raw[lodgement_col].min().date(), "to", df_raw[lodgement_col].max().date())
print("ratings:", sorted(df_raw[rating_col_raw].dropna().unique().tolist()))
df_raw.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/E06000060_certificates.csv'

In [ ]:
# column types and summary stats
df_raw.info()
df_raw.describe(include="all").T.head(20)

In [ ]:
# missing values per column
missing = (df_raw.isna().mean() * 100).sort_values(ascending=False)
missing.head(15)

Main columns I use later include property type, built form, construction age, floor area, heating/walls/roof/window info, co2 emissions and current energy efficiency (the target).

## 3. Analysis

### 3.1 Data Wrangling

In [ ]:
# clean data and save
df = df_raw.copy()
df.columns = df.columns.str.upper().str.strip()

# remove duplicates
id_col = "CERTIFICATE_NUMBER" if "CERTIFICATE_NUMBER" in df.columns else "LMK_KEY"
df["LODGEMENT_DATE"] = pd.to_datetime(df["LODGEMENT_DATE"], errors="coerce")
df = df[df["LODGEMENT_DATE"].between(DATE_START, DATE_END)]
df = df.sort_values("LODGEMENT_DATE").drop_duplicates(id_col, keep="last")

# drop bad or irrelevant columns
drop_cols = [c for c in df.columns if df[c].isna().mean() > 0.5 or c.startswith("ADDRESS")]
drop_cols += [
    "LMK_KEY", "CERTIFICATE_NUMBER", "BUILDING_REFERENCE_NUMBER", "UPRN", "POSTCODE", "POSTTOWN",
    "COUNTY", "CONSTITUENCY", "LOCAL_AUTHORITY", "LOCAL_AUTHORITY_LABEL", "INSPECTION_DATE",
    "LODGEMENT_DATE", "TRANSACTION_TYPE", "REPORT_TYPE", "CURRENT_ENERGY_RATING", "POTENTIAL_ENERGY_RATING",
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

num_cols = df.select_dtypes(include="number").columns
cat_cols = df.select_dtypes(include="object").columns

# clip outliers on numeric columns
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    low, high = df[col].quantile(0.01), df[col].quantile(0.99)
    df[col] = df[col].clip(low, high)

# fill missing values
for col in cat_cols:
    df[col] = df[col].fillna("Unknown")

df = df.dropna(subset=[TARGET])
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

print("after cleaning:", df.shape)
df.to_csv(CLEAN_FILE, index=False)
df.head()

In [ ]:
# feature engineering
model_df = df.copy()

# floor area per room
if "TOTAL_FLOOR_AREA" in model_df.columns and "NUMBER_HABITABLE_ROOMS" in model_df.columns:
    rooms = model_df["NUMBER_HABITABLE_ROOMS"].replace(0, np.nan)
    model_df["FLOOR_AREA_PER_ROOM"] = model_df["TOTAL_FLOOR_AREA"] / rooms
    model_df["FLOOR_AREA_PER_ROOM"] = model_df["FLOOR_AREA_PER_ROOM"].fillna(model_df["FLOOR_AREA_PER_ROOM"].median())

# numeric midpoint from age band
age_map = {
    "England and Wales: before 1900": 1890,
    "England and Wales: 1900-1929": 1915,
    "England and Wales: 1930-1949": 1940,
    "England and Wales: 1950-1966": 1958,
    "England and Wales: 1967-1975": 1971,
    "England and Wales: 1976-1982": 1979,
    "England and Wales: 1983-1990": 1986,
    "England and Wales: 1991-1995": 1993,
    "England and Wales: 1996-2002": 1999,
    "England and Wales: 2003-2006": 2004,
    "England and Wales: 2007 onwards": 2010,
}
if "CONSTRUCTION_AGE_BAND" in model_df.columns:
    model_df["CONSTRUCTION_AGE_MID"] = model_df["CONSTRUCTION_AGE_BAND"].map(age_map)

model_df.to_csv(MODEL_FILE, index=False)
model_df[[TARGET, "FLOOR_AREA_PER_ROOM", "CONSTRUCTION_AGE_MID"]].head()

### 3.2 Descriptive analytics

In [ ]:
# descriptive summary stats
desc_num = model_df.select_dtypes(include="number")
desc_num.describe().T

In [ ]:
# bar chart of epc ratings
rating_col = next((c for c in df_raw.columns if c.lower() == "current_energy_rating"), None)
if rating_col:
    df_raw[rating_col].value_counts().sort_index().plot(kind="bar")
    plt.title(f"EPC rating bands in {LA_NAME}")
    plt.xlabel("Rating")
    plt.ylabel("Count")
    plt.savefig(FIG_DIR / "desc_rating_bar.png")
    plt.show()

In [ ]:
# bar chart of property types
if "PROPERTY_TYPE" in model_df.columns:
    model_df["PROPERTY_TYPE"].value_counts().head(8).plot(kind="bar")
    plt.title("Property types")
    plt.ylabel("Count")
    plt.savefig(FIG_DIR / "desc_property_type_bar.png")
    plt.show()

In [ ]:
# box plots
box_cols = [c for c in [TARGET, "TOTAL_FLOOR_AREA", "CO2_EMISSIONS_CURRENT"] if c in model_df.columns]
model_df[box_cols].plot(kind="box", subplots=True, layout=(1, len(box_cols)), figsize=(12, 4))
plt.suptitle("Box plots of key numeric variables")
plt.savefig(FIG_DIR / "desc_boxplots.png")
plt.show()

### 3.3 Diagnostic analytics

In [ ]:
# diagnostic bar chart by built form
if "BUILT_FORM" in model_df.columns:
    model_df.groupby("BUILT_FORM")[TARGET].mean().sort_values().plot(kind="barh")
    plt.title("Average efficiency score by built form")
    plt.xlabel(TARGET)
    plt.savefig(FIG_DIR / "diag_built_form_bar.png")
    plt.show()

In [ ]:
# diagnostic bar chart by building age
if "CONSTRUCTION_AGE_BAND" in model_df.columns:
    age_order = list(age_map.keys())
    age_means = model_df.groupby("CONSTRUCTION_AGE_BAND")[TARGET].mean()
    age_means = age_means.reindex([a for a in age_order if a in age_means.index])
    age_means.plot(kind="bar")
    plt.title("Average efficiency by construction age")
    plt.ylabel(TARGET)
    plt.xticks(rotation=45, ha="right")
    plt.savefig(FIG_DIR / "diag_age_bar.png")
    plt.show()

In [ ]:
# correlation matrix
corr_cols = [c for c in desc_num.columns if desc_num[c].nunique() > 1][:20]
sns.heatmap(model_df[corr_cols].corr(), cmap="coolwarm")
plt.title("Correlation matrix")
plt.savefig(FIG_DIR / "diag_correlation_matrix.png")
plt.show()

### 3.4 Predictive analytics

Trying to predict the energy efficiency score from the building features. Split 80% train and 20% test.

In [ ]:
# prepare features and train test split
skip = {TARGET, "POTENTIAL_ENERGY_EFFICIENCY", "POTENTIAL_ENERGY_RATING"}
features = [c for c in model_df.columns if c not in skip]

X = model_df[features]
y = model_df[TARGET]

cat_features = X.select_dtypes(include="object").columns.tolist()
num_features = [c for c in X.columns if c not in cat_features]

# impute and one hot encode
prep_steps = [("num", SimpleImputer(strategy="median"), num_features)]
if cat_features:
    prep_steps.append(("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("oh", OneHotEncoder(handle_unknown="ignore", max_categories=12)),
    ]), cat_features))
preprocess = ColumnTransformer(prep_steps)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print("train", len(X_train), "test", len(X_test))

In [ ]:
# train and evaluate models
def run_model(name, model):
    pipe = Pipeline([("prep", preprocess), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    return {
        "model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred),
    }, pipe

lr_results, lr_pipe = run_model("Linear Regression", LinearRegression())
rf_results, rf_pipe = run_model("Random Forest", RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1))

results = pd.DataFrame([lr_results, rf_results]).set_index("model").round(3)
results.to_csv(PROCESSED_DIR / f"{LA_CODE}_model_metrics.csv")
results

In [ ]:
# feature importance plot
rf = rf_pipe.named_steps["model"]
names = num_features.copy()
if cat_features:
    oh = rf_pipe.named_steps["prep"].named_transformers_["cat"].named_steps["oh"]
    names += oh.get_feature_names_out(cat_features).tolist()

importance = pd.Series(rf.feature_importances_, index=names).sort_values(ascending=False).head(15)
importance.sort_values().plot(kind="barh", figsize=(10, 6))
plt.title("Feature importance — factors affecting energy efficiency")
plt.xlabel("Importance")
plt.savefig(FIG_DIR / "feature_importance.png")
plt.show()
importance